In [ ]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
# Change this for each of your 4 notebooks
MODEL_TYPE = "DeepCNNLSTM"  # Options: "MetaCNNLSTM", "DeepCNNLSTM", "TST", "ContrastiveNet"

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs"
FILE_NAME = f"1s3w_mamlpp100_{MODEL_TYPE}_2fcv_hpo"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: 1s3w_mamlpp100_DeepCNNLSTM_2fcv_hpo
From path: C:\Users\kdmen\Repos\pers-gest-cls\optuna_journals\1s3w_mamlpp100_DeepCNNLSTM_2fcv_hpo.log


In [ ]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'mamlpp_pretr_DeepCNNLSTM_2fcv_hpo'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 189 trials.
Best value (Accuracy): 0.9021
Best Hyperparameters:
  meta_batchsize: 4
  maml_inner_steps: 7
  maml_inner_steps_eval: 30
  maml_alpha_init: 0.027719652959239935
  maml_alpha_init_eval: 0.008025773809529157
  outer_lr: 0.00046107285315313145
  wd: 3.623703791716341e-05
  groupnorm_num_groups: 4
  use_GlobalAvgPooling: True
  finetuning_approach: full
  best_or_last_pretr: best
  context_emb_dim: 32
  context_pool_type: mean
  episodes_per_epoch_train: 250
  optimizer: adamw
  use_maml_msl: True


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL
params_to_plot = ["best_or_last_pretr", "context_emb_dim", "context_pool_type", "episodes_per_epoch_train", "finetuning_approach", "groupnorm_num_groups", 
                  "maml_inner_steps", "maml_inner_steps_eval", "maml_msl_num_epochs", "meta_batchsize", "optimizer", "use_GlobalAvgPooling", "use_maml_msl", "wd"]

# PRETRAINING STUFF / MISC
# Apparently finetuning_approach and use_maml_msl only had 1 value? Yes for finetuning currently, but use_maml_msl should have had true and false as well no? ...
params_to_plot_MISC = ["best_or_last_pretr", "finetuning_approach", "use_maml_msl"]

# ARCHITECTURE? --> What does this even mean? How can this vary if we are using the pretrained model....
params_to_plot_ARCH = ["context_emb_dim", "context_pool_type", "groupnorm_num_groups", "use_GlobalAvgPooling"]

# MAML++
params_to_plot_MAMLPP = ["maml_inner_steps", "maml_inner_steps_eval", "maml_msl_num_epochs", "use_maml_msl"]

# LRs MAML++
params_to_plot_LRS = ["maml_alpha_init", "maml_alpha_init_eval", "outer_lr"]

# Learning HPs
params_to_plot_HPS = ["episodes_per_epoch_train", "meta_batchsize", "optimizer", "wd"]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_MISC)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_ARCH)
fig_slice.show()

In [12]:
fig_slice = plot_slice(study, params=params_to_plot_MAMLPP)
fig_slice.show()

In [13]:
fig_slice = plot_slice(study, params=params_to_plot_LRS)
fig_slice.show()

In [14]:
fig_slice = plot_slice(study, params=params_to_plot_HPS)
fig_slice.show()

In [15]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
188,188,0.902083,0.904167,2026-03-26 05:39:54.896533,0 days 00:32:15.365150
183,183,0.901563,0.909375,2026-03-26 04:55:24.222121,0 days 00:22:04.873654
105,105,0.900000,0.891146,2026-03-25 19:12:00.653501,0 days 00:42:12.088314
184,184,0.896875,0.904167,2026-03-26 05:05:31.119002,0 days 00:22:41.578285
187,187,0.892708,0.898958,2026-03-26 05:28:47.239782,0 days 00:24:11.366077
178,178,0.889583,0.902604,2026-03-26 04:13:51.086109,0 days 00:21:47.788550
146,146,0.886458,0.886458,2026-03-26 00:48:38.275137,0 days 00:14:49.006033
172,172,0.885417,0.879687,2026-03-26 03:31:44.608735,0 days 00:13:11.836195
132,132,0.881771,0.890625,2026-03-25 23:10:21.748553,0 days 00:33:52.552828
185,185,0.878646,0.894792,2026-03-26 05:08:32.347378,0 days 00:31:08.010073
